In [ ]:
!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Libraries ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00
Libraries ready


In [ ]:
from groq import Groq
API_KEY = "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"
client = Groq(api_key = API_KEY)
MODEL = "llama-3.1-8b-instant"
print(f'Groq client configured with model:{MODEL}')
print('Make sure API_KEY is replaced with your actual key!')

Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [ ]:
def ask_llm(user_message, system_message="You are a helpful assistaant.", temperature = 0.7, max_tokens = 500):
  response = client.chat.completions.create(
      model = MODEL,
      messages = [
        {
          "role": "system","content": system_message
        },
        {
          "role": "user","content": user_message
        }
      ],
      temperature = temperature,
      max_tokens = max_tokens,

  )

  return response.choices[0].message.content

text_response = ask_llm(
    "Jadeja vs Hardik, who is the best give me the percentage"
)
print('===LLM Response===')
print(text_response)

===LLM Response===
Both Ravindra Jadeja and Hardik Pandya are skilled players in the Indian cricket team. It's subjective to determine who's the best, but I can provide you with some statistics to help compare their performances.

**Batting:**
- Ravindra Jadeja (Tests): Average - 35.17, SR - 55.35
- Ravindra Jadeja (ODIs): Average - 30.35, SR - 92.55
- Ravindra Jadeja (T20Is): Average - 25.41, SR - 123.38
- Hardik Pandya (Tests): Average - 29.33, SR - 82.45 (Note: Hardik Pandya has played limited Tests)
- Hardik Pandya (ODIs): Average - 33.45, SR - 126.42
- Hardik Pandya (T20Is): Average - 29.14, SR - 148.19

**Bowling:**
- Ravindra Jadeja (Tests): Economy - 3.11, Strike rate - 83.42
- Ravindra Jadeja (ODIs): Economy - 4.69, Strike rate - 44.83
- Ravindra Jadeja (T20Is): Economy - 7.43, Strike rate - 24.55
- Hardik Pandya (Tests): Economy - 4.44, Strike rate - 53.5
- Hardik Pandya (ODIs): Economy - 4.59, Strike rate - 43.33
- Hardik Pandya (T20Is): Economy - 8.51, Strike rate - 24.33



In [ ]:
response_etl = ask_llm(
    "In 3 bullet points, explain hoow the Medallion Architecture"
    "(Bronze, Silver, Gold layers) relates to ETL Pipelines.",
    system_message = "You are a senior data engineering instructor."
                      "Be Consise and practical."
)
print('Medallion +  ETL Connection')
print(response_etl)
print()
print('---Token explanation---')
print('Each work is roughly 1- 2 tokens')
print('The Model above used approximately', len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(~)')

Medallion +  ETL Connection
Here's how the Medallion Architecture relates to ETL Pipelines in 3 bullet points:

• **Bronze Layer (Ingest Layer)**: This is the first layer where raw data is ingested from various sources (databases, APIs, files) into a structured format, often using tools like Apache NiFi, Apache Kafka, or AWS Kinesis. The goal is to load the data into a data lake or a data warehouse.

• **Silver Layer (Transform Layer)**: This layer transforms the raw data into a more usable format, applying business logic and data quality checks as needed. It's where ETL (Extract, Transform, Load) pipelines are typically implemented, using tools like Apache Spark, Apache Beam, or AWS Glue. The transformed data is then stored in a data mart or a data warehouse.

• **Gold Layer (Warehouse Layer)**: This is the final layer where the transformed data is stored in a data warehouse, optimized for querying and analysis. The Gold Layer is where business users can access the data, using tools l

In [ ]:
zero_shot_response = ask_llm(
    "Extract the city name from this address:"
    "456 Brigade Road, Bangalore 560025, Karnataka India"
)
print('Zero-Shot Result:')
print(zero_shot_response)
print()
ambiguous_response = ask_llm("Clean this data: ramesh kumar, 45000, mumbai")
print('Ambiguous Zero-Shot Result:')
print(ambiguous_response)
print()
print('Problem: output format is unpredictable and not machine-parseable')

Zero-Shot Result:
The city name is: Bangalore

Ambiguous Zero-Shot Result:
After analyzing the given data, I've identified the following issues:

- The data contains a name ("ramesh kumar") which is not numerical and doesn't seem to fit any specific category.
- The data contains a salary ("45000") which is numerical but might not be in the correct format (e.g., it's missing commas for thousands).
- The data contains a location ("mumbai") which is a string but doesn't seem to fit any specific category.

To clean this data, I would suggest the following:

- **Name:** Keep the name as it is, since it's a unique identifier.
- **Salary:** Round the salary to the nearest thousand and add commas for better readability. In this case, the salary would be "45,000".
- **Location:** Keep the location as it is, since it's a string and might be useful for categorization or filtering.

The cleaned data would look like this:
```markdown
Name: ramesh kumar
Salary: 45,000
Location: mumbai
```
Or, if you

In [ ]:
few_shot_prompt = """
  Conver exployee text to json. Here are examples:
  Input: Ramesh Kumar, 45000, mumbai
  Output:{"name": "Ramesh Kumar", "salary": 45000, "city": "mumbai"}

  Input: priya nair, 52000, Delhi
  Output:{"name": "Priya Nair", "salary": 52000, "city": "Delhi"}
"""
few_shot_response = ask_llm(
    few_shot_prompt,
    system_message="You are an assistant that converts employee text to JSON. Only output the JSON and nothing else.",
    temperature = 0.0
)
print('Few-shot Result:')
print(few_shot_response)
print()

try:
  parsed = json.loads(few_shot_response.strip())
  print('Successfully parsed JSON')
  print(f'Name: {parsed["name"]}, Salary: {parsed["salary"]}, City: {parsed["city"]}')
except json.JSONDecodeError:
  print('Parsing failed - model added extra text')
  print('Solution: add explicit instructions in the system prompt')


Few-shot Result:
{"name": "Ramesh Kumar", "salary": 45000, "city": "mumbai"}

Successfully parsed JSON
Name: Ramesh Kumar, Salary: 45000, City: mumbai


In [ ]:
same_question = """Review the following Python code for potential issues:\n\ndef process_data(data_list):\n    result = []\n    for item in data_list:\n        result.append(item * 2)\n    return result\n\nprint(process_data([1, 2, '3']))\n"""
generic_response = ask_llm(same_question, temperature = 0.2)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()

role_response = ask_llm(
    same_question,
    system_message = "You are a senior data engineer with 10 years of production."
                      "experience. Review code critically for production readiness,"
                      "data type issues, and potential failures at scale.",
    temperature = 0.2
)
print('With Role Prompting:(Senior Data Engineer)')
print(role_response[:400],'...')
print()
print('Notice: role prompting produces more technical, actionable feedback')


Without Role Prompting:
**Code Review**

The provided Python code appears to be a simple function that takes a list of data, doubles each item, and returns the resulting list. However, there are a few potential issues to consider:

### 1. Type Hints

The function `process_data` lacks type hints for its parameters and retur ...

With Role Prompting:(Senior Data Engineer)
**Code Review**

The provided Python code appears to be a simple data processing function that doubles each element in a list. However, there are several potential issues that need to be addressed for production readiness:

### 1. Data Type Issues

The code does not handle different data types within the input list. When it encounters a string ('3'), it attempts to multiply it by 2, which results  ...

Notice: role prompting produces more technical, actionable feedback


In [ ]:
prompt = "Give me one creative name for a data analytics startup"

print('===Temperature Experiment===')
for temp in[0.0, 0.5, 1.0]:
  response = ask_llm(prompt, temperature = temp)
  print(f'Temperature {temp}: {response.strip()}')
  time.sleep(1)

  print()
  print('Observation:')
  print('temperature = 0.0 -> Same or very similar answer every run (deterministic)')
  print('temperature = 0.5 -> some variations')
  print('temperature = 1.0'-> more creative)

SyntaxError: invalid syntax (3797863159.py, line 13)